# T2SMark baseline 结果 Colab Notebook

该 Notebook 在独立 Colab 冷启动会话中运行 T2SMark 外部 baseline 本体, 产出 T2SMark 原生 `results.json`。该文件会保存到 Google Drive 的 `external_baseline_inputs/t2smark/results.json`, 供 `paper_workflow/colab_external_baseline_outputs.ipynb` 调用 CEG adapter 转换为统一的 `baseline_observations.json`。

该 Notebook 不实现 CEG 主方法, 不生成 CEG 水印, 不运行 CEG detection, 也不调用 CEG-WM。T2SMark 作为第三方外部 baseline 在 Colab 本地运行, 其源码克隆到 Colab 临时目录, 不进入 CEG 主方法目录。


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")
DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
LOCAL_RUNTIME_ROOT = Path("/content/ceg_runtime")

T2SMARK_REPO_URL = "https://github.com/0xD009/T2SMark.git"
T2SMARK_REPO_ROOT = Path("/content/external_baselines/t2smark/source")

PROMPT_PLAN_PROFILE = "paper_main_probe"
RUN_ID = f"{PROMPT_PLAN_PROFILE}_t2smark_baseline_outputs"
WORKSPACE = LOCAL_RUNTIME_ROOT / RUN_ID
RESET_LOCAL_RUNTIME_WORKSPACE = True

PROMPT_PLAN = REPO_ROOT / "prompts" / "prompt_plans" / f"{PROMPT_PLAN_PROFILE}_prompt_plan.json"
T2SMARK_PROMPT_CSV = WORKSPACE / "inputs" / "t2smark_prompts.csv"
T2SMARK_OUTPUT_ROOT = WORKSPACE / "t2smark_outputs"
T2SMARK_RUN_NAME = RUN_ID
T2SMARK_RUN_OUTPUT_DIR = T2SMARK_OUTPUT_ROOT / T2SMARK_RUN_NAME
T2SMARK_RESULTS_LOCAL = T2SMARK_RUN_OUTPUT_DIR / "results.json"
T2SMARK_RESULTS_DRIVE = DRIVE_ROOT / "external_baseline_inputs" / "t2smark" / "results.json"

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
INSTALL_T2SMARK_DEPENDENCIES = True
RUN_T2SMARK_BACKEND = True
ARCHIVE_T2SMARK_OUTPUTS = True
SAVE_T2SMARK_IMAGES = False
T2SMARK_SEED = 0
T2SMARK_GUIDANCE_SCALE = 4.0
T2SMARK_NUM_INFERENCE_STEPS = 40
T2SMARK_NUM_INVERSION_STEPS = 10
T2SMARK_CLIP_TEST_NUM = 0
T2SMARK_START_INDEX = 0
T2SMARK_FIX_KEY = False
MAX_PROMPTS = None


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"非 Colab 环境或 Drive 已挂载: {exc}")

if not REPO_ROOT.exists():
    clone_cmd = ["git", "clone"]
    if REPO_BRANCH:
        clone_cmd += ["--branch", REPO_BRANCH]
    clone_cmd += [REPO_URL, str(REPO_ROOT)]
    print("运行:", " ".join(clone_cmd))
    subprocess.run(clone_cmd, check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--all", "--prune"], check=True)
    if REPO_BRANCH:
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

if not T2SMARK_REPO_ROOT.exists():
    T2SMARK_REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    clone_t2smark_cmd = ["git", "clone", T2SMARK_REPO_URL, str(T2SMARK_REPO_ROOT)]
    print("运行:", " ".join(clone_t2smark_cmd))
    subprocess.run(clone_t2smark_cmd, check=True)
else:
    subprocess.run(["git", "-C", str(T2SMARK_REPO_ROOT), "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", str(T2SMARK_REPO_ROOT), "pull", "--ff-only"], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)

if INSTALL_T2SMARK_DEPENDENCIES:
    requirements = T2SMARK_REPO_ROOT / "requirements.txt"
    if requirements.is_file():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "diffusers==0.32.0",
            "transformers",
            "accelerate",
            "safetensors",
            "datasets",
            "open_clip_torch",
            "scikit-learn",
            "pandas",
            "torchvision",
        ],
        check=True,
    )

from paper_workflow.colab_utils.runtime import archive_directory_to_drive, write_json

if WORKSPACE.exists() and RESET_LOCAL_RUNTIME_WORKSPACE:
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)
T2SMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("repo_root =", REPO_ROOT)
print("t2smark_repo_root =", T2SMARK_REPO_ROOT)
print("workspace =", WORKSPACE)
print("prompt_plan =", PROMPT_PLAN, "exists=", PROMPT_PLAN.exists())
print("t2smark_results_drive =", T2SMARK_RESULTS_DRIVE)


In [ ]:
if not PROMPT_PLAN.is_file():
    raise FileNotFoundError(f"仓库内置 prompt plan 不存在: {PROMPT_PLAN}")

prompt_plan = json.loads(PROMPT_PLAN.read_text(encoding="utf-8-sig"))
prompt_rows = list(prompt_plan.get("prompts", []))
if MAX_PROMPTS is not None:
    prompt_rows = prompt_rows[: int(MAX_PROMPTS)]
if not prompt_rows:
    raise ValueError("prompt plan 中没有可供 T2SMark 使用的 prompts")

T2SMARK_PROMPT_CSV.parent.mkdir(parents=True, exist_ok=True)
with T2SMARK_PROMPT_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["Our GT caption", "prompt_id", "image_id", "split", "sample_index"])
    writer.writeheader()
    for index, row in enumerate(prompt_rows, start=1):
        writer.writerow(
            {
                "Our GT caption": row.get("prompt_text", ""),
                "prompt_id": row.get("prompt_id", f"prompt_{index:06d}"),
                "image_id": row.get("image_id", f"image_{index:06d}"),
                "split": row.get("split", "test"),
                "sample_index": row.get("sample_index", index),
            }
        )

T2SMARK_SAMPLE_COUNT = len(prompt_rows)
prompt_manifest = {
    "artifact_name": "t2smark_prompt_alignment_manifest.json",
    "producer_id": "colab_t2smark_baseline_outputs",
    "prompt_plan_path": str(PROMPT_PLAN),
    "prompt_plan_profile": PROMPT_PLAN_PROFILE,
    "t2smark_prompt_csv": str(T2SMARK_PROMPT_CSV),
    "prompt_count": T2SMARK_SAMPLE_COUNT,
    "csv_prompt_column": "Our GT caption",
    "alignment_rule": "T2SMark results numeric keys follow the same order as CEG prompt_plan.prompts",
}
write_json(WORKSPACE / "t2smark_prompt_alignment_manifest.json", prompt_manifest)
print(json.dumps(prompt_manifest, ensure_ascii=False, indent=2))


In [ ]:
t2smark_cmd = [
    sys.executable,
    "run_sd35.py",
    "--name",
    T2SMARK_RUN_NAME,
    "--output_dir",
    str(T2SMARK_OUTPUT_ROOT),
    "--dataset_key",
    str(T2SMARK_PROMPT_CSV),
    "--model_key",
    MODEL_ID,
    "--SDv35M",
    "--guidance_scale",
    str(T2SMARK_GUIDANCE_SCALE),
    "--num_inference_steps",
    str(T2SMARK_NUM_INFERENCE_STEPS),
    "--num_inversion_steps",
    str(T2SMARK_NUM_INVERSION_STEPS),
    "--robust_test_num",
    str(T2SMARK_SAMPLE_COUNT),
    "--clip_test_num",
    str(T2SMARK_CLIP_TEST_NUM),
    "--start_idx",
    str(T2SMARK_START_INDEX),
    "--seed",
    str(T2SMARK_SEED),
]
if SAVE_T2SMARK_IMAGES:
    t2smark_cmd.append("--save_image")
if T2SMARK_FIX_KEY:
    t2smark_cmd.append("--fix_key")

run_manifest = {
    "artifact_name": "t2smark_backend_run_manifest.json",
    "producer_id": "colab_t2smark_baseline_outputs",
    "external_baseline_id": "t2smark",
    "t2smark_repo_url": T2SMARK_REPO_URL,
    "t2smark_repo_root": str(T2SMARK_REPO_ROOT),
    "command": t2smark_cmd,
    "working_directory": str(T2SMARK_REPO_ROOT),
    "model_id": MODEL_ID,
    "prompt_csv": str(T2SMARK_PROMPT_CSV),
    "expected_results_path": str(T2SMARK_RESULTS_LOCAL),
    "drive_results_path": str(T2SMARK_RESULTS_DRIVE),
    "prompt_count": T2SMARK_SAMPLE_COUNT,
    "formal_result_claim": False,
}
write_json(WORKSPACE / "t2smark_backend_run_manifest.json", run_manifest)
print("T2SMark 命令:")
print(" ".join(t2smark_cmd))

if RUN_T2SMARK_BACKEND:
    subprocess.run(t2smark_cmd, cwd=str(T2SMARK_REPO_ROOT), check=True)
else:
    print("RUN_T2SMARK_BACKEND = False, 仅打印命令并写出 manifest。")

if RUN_T2SMARK_BACKEND and not T2SMARK_RESULTS_LOCAL.is_file():
    raise FileNotFoundError(f"T2SMark results.json 未生成: {T2SMARK_RESULTS_LOCAL}")


In [ ]:
if RUN_T2SMARK_BACKEND:
    T2SMARK_RESULTS_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(T2SMARK_RESULTS_LOCAL, T2SMARK_RESULTS_DRIVE)
    result_payload = json.loads(T2SMARK_RESULTS_DRIVE.read_text(encoding="utf-8-sig"))
    numeric_result_count = sum(1 for key, value in result_payload.items() if str(key).isdigit() and isinstance(value, dict))
    acceptance_report = {
        "artifact_name": "t2smark_baseline_output_acceptance_report.json",
        "overall_decision": "pass" if numeric_result_count == T2SMARK_SAMPLE_COUNT else "fail",
        "producer_id": "colab_t2smark_baseline_outputs",
        "results_local_path": str(T2SMARK_RESULTS_LOCAL),
        "results_drive_path": str(T2SMARK_RESULTS_DRIVE),
        "expected_prompt_count": T2SMARK_SAMPLE_COUNT,
        "numeric_result_count": numeric_result_count,
        "contains_tpr": "tpr" in result_payload,
        "contains_bit_accuracy": "bit_accuracy" in result_payload,
    }
    write_json(WORKSPACE / "t2smark_baseline_output_acceptance_report.json", acceptance_report)
    if acceptance_report["overall_decision"] != "pass":
        raise RuntimeError(f"T2SMark 结果数量不符合 prompt plan: {acceptance_report}")
    print(json.dumps(acceptance_report, ensure_ascii=False, indent=2))
else:
    print("RUN_T2SMARK_BACKEND = False, 跳过 results.json 复制和验收。")


In [ ]:
if ARCHIVE_T2SMARK_OUTPUTS and T2SMARK_RUN_OUTPUT_DIR.is_dir():
    archive_manifest = archive_directory_to_drive(
        source_root=T2SMARK_RUN_OUTPUT_DIR,
        drive_root=DRIVE_ROOT,
        archive_group="t2smark_baseline_outputs",
        run_id=RUN_ID,
        archive_name=RUN_ID,
    )
    print("t2smark_archive_manifest =")
    print(json.dumps(archive_manifest, ensure_ascii=False, indent=2))
else:
    print("未归档 T2SMark 输出目录。ARCHIVE_T2SMARK_OUTPUTS =", ARCHIVE_T2SMARK_OUTPUTS)

paths = {
    "prompt_csv": T2SMARK_PROMPT_CSV,
    "results_local": T2SMARK_RESULTS_LOCAL,
    "results_drive": T2SMARK_RESULTS_DRIVE,
    "run_manifest": WORKSPACE / "t2smark_backend_run_manifest.json",
    "prompt_alignment_manifest": WORKSPACE / "t2smark_prompt_alignment_manifest.json",
    "acceptance_report": WORKSPACE / "t2smark_baseline_output_acceptance_report.json",
    "drive_archive": DRIVE_ROOT / "archives" / "t2smark_baseline_outputs" / f"{RUN_ID}.zip",
}
for name, item_path in paths.items():
    print(name, "=", item_path, "exists=", item_path.exists())
